# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, examine, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset for Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load [metadata](https://mlcommons.org/croissant/latest/metadata.html) and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Pretty print metadata title and description
print("Dataset Title:", metadata.name)
print("Description:\n", metadata.description)


## 2. Data Overview
Review available Record Sets (tables), their fields and columns, and inspect their `@id`s. In Croissant, **everything** (datasets, record sets, fields, columns, etc) is uniquely identified by their `@id`.

Let's list all `RecordSet` objects in the dataset.

In [ ]:
# List all record sets and their fields

print("Available Record Sets:")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"- Record Set label: {rs.label}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List fields under this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.label} (field @id: {field.id}) -- type: {field.data_type}, column: {field.column.id if field.column else 'N/A'}")
    print()

# Store IDs for further use
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s from above.

We will extract all main record sets for demonstration.

In [ ]:
# Extract all records from every record set as a DataFrame
dfs = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set @id={record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dfs[record_set_id].columns.tolist()}")
        print(f"  Preview:\n{dfs[record_set_id].head()}\n")
    else:
        print("  No records found for this record set.\n")
# For the rest of the notebook, select the first non-empty record set for demonstration
for rid in record_set_ids:
    if rid in dfs:
        main_record_set_id = rid
        break

print(f"Selected record set for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Common data processing examples: filtering records, normalizing a numeric field, and grouping data by a categorical field.

- Select a **numeric field** (`@id`)
- Apply filters and normalization
- Optionally group by a category

Below we auto-detect a numeric field to demonstrate. Replace variables as needed for your own use.

In [ ]:
import numpy as np

# Identify numeric fields (try float/int columns that are not IDs)
df = dfs[main_record_set_id]
numeric_field = None
for col in df.columns:
    # Heuristic: try to coerce to float and check proportion of NaNs
    try:
        s = pd.to_numeric(df[col], errors='coerce')
        if s.notna().mean() > 0.8:  # majority is numeric
            numeric_field = col
            break
    except Exception:
        continue
if numeric_field is None:
    raise ValueError('No obvious numeric field found in the first record set.')

print(f"Using numeric field for analysis: {numeric_field}")
# Ensure correct dtype
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Set a threshold at 75th percentile as example
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (75th percentile):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical column (if available)
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() < 10:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped filtered records by '{group_field}', mean {numeric_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field before and after filtering, and plot normalized values.

(*Requires matplotlib for visualization*)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.subplot(1,2,1)
df[numeric_field].hist(bins=30, alpha=0.7)
plt.title(f"Distribution of {numeric_field} (all records)")
plt.xlabel(numeric_field)
plt.ylabel("Count")

plt.subplot(1,2,2)
filtered_df[f"{numeric_field}_normalized"].hist(bins=30, alpha=0.7, color='orange')
plt.title(f"Distribution of {numeric_field}_normalized (filtered)")
plt.xlabel(f"{numeric_field}_normalized")
plt.tight_layout()
plt.show()


## 6. Conclusion

This notebook demonstrated how to:
- Load Croissant-compliant datasets using the [`mlcroissant`](https://mlcommons.org/croissant/) library.
- Inspect metadata to find record sets and fields **using Croissant `@id`s**.
- Load tabular data from record sets, process numeric fields, and visualize distributions.

This approach ensures FAIRness and reproducibility by referencing all entities/modules via their semantic ontology IDs (`@id`s).

For deeper analysis, consult the [Croissant and mlcroissant documentation](https://mlcommons.org/croissant/latest/).
